# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

## Dataset Source
The dataset is provided via a Croissant schema JSON-LD URL and contains structured survey results about adoption of indigenous and modern knowledge for rangeland management in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and prepare to access record sets and tabular data using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata attributes
md = dataset.metadata
print(f"Dataset Name: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Authors: {md.author}")
print(f"Identifier: {md.identifier}")

## 2. Data Overview

List available record sets and fields (by their `@id`). This gives an overview of the main tables and variables present in the Croissant dataset.

In [ ]:
# List all record sets available
print("Available Record Sets (@id):")
for record_set in dataset.record_sets:
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name', '[no name]')}")

# For each record set, list its fields (columns)
for record_set in dataset.record_sets:
    print(f"\nFields in record set '@id': {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = field.get('@id', str(field))
            name = field.get('name', '[no name]')
            print(f"  - Field @id: {field_id} name: {name}")

## 3. Data Extraction

Load data from available record sets into Pandas DataFrames for analysis. All extraction uses the entity `@id`.

*(Choose record set `@id`s you want to load. Here we load all found in overview above; you may customize by selecting a subset.)*

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Records is an iterator of dicts for the record set
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    else:
        print("- No data loaded for this record set.")

# Show columns of the first non-empty DataFrame and its head
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in first DataFrame ({first_rs}): {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())
else:
    print("No record sets found with data.")

## 4. Exploratory Data Analysis (EDA)

Apply some general EDA: filtering numeric values, normalization, and grouping, referencing all fields by their `@id`.

In [ ]:
# Select a record set for EDA (replace with your target if known)
record_set_id = list(dataframes.keys())[0] if dataframes else None

if record_set_id is not None:
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    
    # Find numeric columns by attempting conversion
    sample_row = df.iloc[0] if len(df) else None
    numeric_field_id = None
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna())
            numeric_field_id = col
            break
        except Exception:
            continue
    
    if numeric_field_id:
        print(f"Numeric field identified (@id): {numeric_field_id}")
        # Filter out values less than a threshold (arbitrarily 10)
        df_num = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = 10
        filtered_df = df[df_num > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (df_num[filtered_df.index] - df_num.mean()) / df_num.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("No numeric fields found for EDA in this record set.")

    # Try to find a non-numeric field for grouping
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field = col
            break
    if group_field and numeric_field_id:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped normalized data by {group_field}:")
        display(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field and the effect of grouping/categorization. All axes are annotated with field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    # Plot histogram
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.show()
    
    # If grouping field exists, plot means by group
    if group_field and numeric_field_id in filtered_df.columns:
        means_by_group = filtered_df.groupby(group_field)[numeric_field_id].mean()
        means_by_group.plot(kind='bar', figsize=(8,4))
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field} (Filtered)")
        plt.show()

## 6. Conclusion

- The dataset provides regression and socio-demographic data for rangeland management knowledge adoption.
- Record sets and fields can be explored programmatically via `@id`.
- For further analysis, tailor EDA by consulting field descriptions in the Croissant schema, and conduct domain-specific filtering and visualization as needed.

For more details about the dataset structure, fields, and licensing, see the [online schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) and [mlcroissant documentation](https://mlcroissant.readthedocs.io/).